# Run the bin model on your campus
**Part of "What's in the Bin? A Student-Run Plastic Pilot"** (Fresno State, BEAM Circular / CBIO Collaborative challenge)

Type your campus name, then **Runtime > Run all**. In a few minutes you get:
1. a map of where waste-generating foot traffic is on your campus (estimated from OpenStreetMap: building size and use),
2. the spots where a small number of bin stations would reach the most people, and
3. if you upload a list of your bins: your bins ranked by the foot traffic each one serves, so you know where recycling and clear signs matter most.

No account, no cost, no installation. **Honesty note:** foot traffic is an estimate, so results are *projected* reach, not measured recycling. A pilot like ours replaces the estimates with real counts.

## 1. Your campus
Type the name the way you'd search for it on a map, with the city and state. Examples:
`University of the Pacific, Stockton, California` · `Modesto Junior College, College Avenue, Modesto` · `California State University, Stanislaus`

**Optional:** to rank your own bins, upload a CSV (left sidebar: folder icon, then upload) with `lat` and `lon` columns, like `templates/bins_template.csv`, and put its file name below.

In [ ]:
CAMPUS = "University of the Pacific, Stockton, California"
BINS_CSV = None          # e.g. "my_bins.csv" after uploading it; None = suggest spots only
N_SITES = 10             # how many bin spots to suggest
RADIUS_M = 50            # walking reach in meters (about a 40-second walk)

## 2. Setup (downloads our open-source code)

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    %cd /content
    !rm -rf /content/repo && git clone -q https://github.com/Rondo12/CoverageBinCA---CBIO-Collaborative-Innovation-Design-Challenge.git /content/repo
    %cd /content/repo
    !pip install -q -e . osmnx
    if BINS_CSV and not os.path.isabs(BINS_CSV):
        BINS_CSV = os.path.join("/content", BINS_CSV)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Ready in", os.getcwd())

## 3. Run it
This finds the campus on OpenStreetMap, downloads its buildings, walkways, parking and food spots, estimates foot traffic, and runs the model. It usually takes 1 to 3 minutes.

In [ ]:
from binopt.anycampus import run, campus_map
result = run(CAMPUS, bins_csv=BINS_CSV, n_sites=N_SITES, radius_m=RADIUS_M)

## 4. The map
Gray circles = estimated foot traffic (bigger = busier). Blue = suggested bin spots. Red = your busiest bins (if you uploaded them). Switch layers on and off at the top right.

In [ ]:
campus_map(result)

## 5. The numbers

In [ ]:
print(f"{result.name}: {result.area_km2} km2 studied")
print(f"{N_SITES} suggested spots reach about {result.suggested_share:.0%} of estimated foot traffic within {RADIUS_M} m")
display(result.suggested.rename(columns={"near": "nearest named building"}))
if result.ranked is not None:
    print("Your bins, busiest first:")
    display(result.ranked[["rank", "bin_id", "share", "serves"]].head(10).style.format({"share": "{:.1%}"}).hide(axis="index"))

## What to do with it
- **No bin list yet?** The suggested spots show where a recycling pilot would reach the most people. Walk them and check what bins are already there.
- **Have a bin list?** Start with the top-ranked bins: that's where adding recycling and clear "no bags in recycling" signs reaches the most people.
- **Want real numbers?** Run a weekly bin log at a few top-ranked sites (our form is in `scripts/create_weekly_log_form.gs`) and track which single-use plastic items end up in the wrong bin. That turns estimates into decisions.

Map data © OpenStreetMap contributors (ODbL). The code is open source in this repository.